# Study 830 — BAB Across Asset Classes ⚖️🌐

**Does "betting against beta" work when the assets are whole *asset classes*, not stocks?**

Frazzini & Pedersen (2014) show that low-beta stocks earn too much per unit of risk and
high-beta stocks too little — a *flat* security-market line — and package the edge as the
**BAB factor**: long low-beta (levered to unit beta), short high-beta (de-levered to unit
beta), so the book is beta-neutral. The famous claim is that this flat SML is **everywhere**,
including *across asset classes*. We build the multi-asset version on nine liquid asset-class
ETFs (2007-04-11 → 2026-06-30): equities (SPY/EFA/EEM), bonds & credit (TLT/LQD/HYG), gold
(GLD), commodities (DBC) and REITs (VNQ); beta is measured to their equal-weight portfolio.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. The 'market' is a fixed current-membership ETF basket — named on the
Signal axis.*


## 1. The idea in one picture

The CAPM says higher beta should pay higher return in a straight line. In practice the line is too **flat**: leverage-constrained investors crowd into high-beta assets (bidding their risk-adjusted return down) and shun low-beta ones. BAB harvests that: buy the calm assets and *lever them up* to market risk, sell the racy assets and *scale them down*, so market swings cancel and what is left is the flat-SML alpha. The bet here is that this works not just among stocks but among **asset classes**.

In [1]:
import numpy as np, pandas as pd
R = dict(bab_bps=0.54, t_nw=0.31, alpha_bps=2.86, alpha_t=1.61, realized_beta=-0.83, sharpe=0.06)
print('multi-asset BAB factor: %+.2f bps/day  (Newey-West t = %+.2f)'
      % (R['bab_bps'], R['t_nw']))
print('  CAPM alpha %+.2f bps/day (HAC t = %+.2f); realized market beta %+.2f'
      % (R['alpha_bps'], R['alpha_t'], R['realized_beta']))
print('  gross annualized Sharpe: %.2f' % R['sharpe'])

multi-asset BAB factor: +0.54 bps/day  (Newey-West t = +0.31)
  CAPM alpha +2.86 bps/day (HAC t = +1.61); realized market beta -0.83
  gross annualized Sharpe: 0.06


## 2. Is the sort just lucky? A live synthetic control

We plant a flat-SML premium in a seeded toy world (`edge>0`: low-beta assets carry a positive alpha, high-beta a negative one) and check the detector recovers it — and stays *silent* on the null (`edge=0`, CAPM holds, betas still disperse). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from bab_multiasset import data, strategy as st
null = st.synthetic_detect(data.synthetic_series(edge=0.0, seed=830, n_days=2500))
planted = st.synthetic_detect(data.synthetic_series(edge=0.0006, seed=830, n_days=2500))
print('null world   : BAB NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: BAB NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : BAB NW t = -0.42  (should be ~0)
planted world: BAB NW t = +3.55  (should light up)


## 3. The honest verdict — the multi-asset BAB does *not* deliver an alpha here

On nine liquid asset classes the beta-neutral BAB factor earns **+0.54 bps/day** with Newey-West *t* = **+0.31** — indistinguishable from zero. Even measured as a CAPM alpha it is only **+2.86 bps/day (t = +1.61)**, again short of significance, and the book's *realized* market beta is **-0.83** — far from neutral: the 'low-beta' long leg is dominated by Treasuries and gold, so multi-asset BAB is really a disguised **long-duration / short-equity** tilt. It worked weakly before 2016 (alpha *t* = +1.95) and reversed after (*t* = -0.25). The seeded synthetic control recovers a *planted* flat-SML premium cleanly (*t* = +3.55), so the machinery works — the multi-asset SML just is not flat enough to trade. **Signal: None. Tradability: Mirage.**